# Water level prediction using ERA5 (CNN 1D)

This notebook implements a 1D CNN model in Keras to predict hourly water level at 5000 nodes using:
- Past ephemerides (6 features)
- Past ERA5 variables (3 variables on a 5x9 grid)
- Node latitude/longitude

**Important constraint**: the model does **not** use past water level values.

In [1]:
import gdown
# train data
!gdown 1Ncexf_vB55cpiCeNr-hIRrdpquYaav6B
!gdown 1THbGvO9mVjg_wfZTabRbQBOVpaECl3my
!gdown 16M1zB54PKkKS6SK8W_U83UURWu1T_AxR
!gdown 161OYs8KQSn3RrXezCNwFvOewXLJe5wBf
# test data
!gdown 1iwqd4xzHc98OYqBpGsuUW4SG5AQDtdNR #(8784, 5000)  hours,nodes water levels
!gdown 1cHqyeXtmaiC_3v9uadMD7Y0hONGf2R1q
!gdown 1AoFAD2viMarikhU5b5Etdsklx08EzKKb
!gdown 1sWoTlJih-mqDdP9TBzhyXTqHHS5Jr9fe
# latitude and longitude of nodes
!gdown 1Mg52QAIo4bfpzJF0dsI8mpZf09tHTzj8
!gdown 1wWz0EWbGiBkZ0vfJD8KeZkmVZfmRiYLk

Failed to retrieve file url:

	Too many users have viewed or downloaded this file recently. Please
	try accessing the file again later. If the file you are trying to
	access is particularly large or is shared with many people, it may
	take up to 24 hours to be able to view or download the file. If you
	still can't access a file after 24 hours, contact your domain
	administrator.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1Ncexf_vB55cpiCeNr-hIRrdpquYaav6B

but Gdown can't. Please check connections and permissions.
Downloading...
From: https://drive.google.com/uc?id=1THbGvO9mVjg_wfZTabRbQBOVpaECl3my
To: /content/dist_alt_az_moon-sun_coord13-45_2010-2019_norm.npy
100% 4.21M/4.21M [00:00<00:00, 253MB/s]
Downloading...
From: https://drive.google.com/uc?id=16M1zB54PKkKS6SK8W_U83UURWu1T_AxR
To: /content/ERA5_adriatic_u10v10sp_2010-2019.npy
100% 94.7M/94.7M [00:01<00:00, 53.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=161OYs8KQSn3

## Utilities

In [125]:
def get_era5_coord(lat, lon):
    era5_row, era5_col = 5, 9
    lat_min, lat_max = 44.94972, 45.8
    lon_min, lon_max = 12.12863, 13.81283

    delta_lat = lat_max - lat_min
    delta_lon = lon_max - lon_min

    lon_coord = np.ceil((lon - lon_min) / delta_lon * (era5_col - 1))
    lat_coord = 4 - np.ceil((lat - lat_min) / delta_lat * (era5_row - 1))

    return int(lat_coord), int(lon_coord)

def RMSE(wl_true, wl_pred):
    return np.sqrt(np.mean(np.square(wl_pred - wl_true)))

## Build dataset

We create samples using a fixed lookback window (e.g. 48 hours). For each node and each time step, we build an input tensor of shape:

`[lookback, features]` where features include:
- 6 ephemerides
- 3 ERA5 variables at the node's nearest ERA5 grid point
- 2 node coords (lat, lon)

**No water level history is used.**

In [126]:
LOOKBACK = 48
N_NODES = 5000

# normalize ERA5 variables (train only)
era5_flat = train_era5.reshape(3, -1)
era5_mean = era5_flat.mean(axis=1, keepdims=True)
era5_std = era5_flat.std(axis=1, keepdims=True) + 1e-6

def normalize_era5(arr):
    arr = arr.copy()
    for i in range(3):
        arr[i] = (arr[i] - era5_mean[i]) / era5_std[i]
    return arr

train_era5_norm = normalize_era5(train_era5)
test_era5_norm = normalize_era5(test_era5)

In [ ]:
def make_dataset(ephem, era5, wl, lat_vec, lon_vec, lookback=48, max_nodes=1000):
    """
    Build dataset for a subset of nodes to fit in memory.
    """
    n_time = wl.shape[0]
    node_indices = np.random.choice(np.arange(len(lat_vec)), size=max_nodes, replace=False)

    X_list = []
    y_list = []
    for node in tqdm(node_indices):
        r, c = get_era5_coord(lat_vec[node], lon_vec[node])
        node_lat = lat_vec[node]
        node_lon = lon_vec[node]
        for t in range(lookback, n_time):
            ephem_t = ephem[:, t-lookback:t].T  # (lookback, 6)
            era5_t = era5[:, t-lookback:t, r, c].T  # (lookback, 3)
            coords = np.tile([node_lat, node_lon], (lookback, 1))
            x = np.concatenate([ephem_t, era5_t, coords], axis=1)  # (lookback, 11)
            X_list.append(x)
            y_list.append(wl[t, node])

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y

: 

## Build training data

In [128]:
# To keep memory manageable, sample subset of nodes for training
X_train, y_train = make_dataset(
    train_ephem,
    train_era5_norm,
    train_wl,
    lat_vec,
    lon_vec,
    lookback=LOOKBACK,
    max_nodes=800
)

print(X_train.shape, y_train.shape)

  3%|▎         | 22/800 [00:24<15:27,  1.19s/it]

: 

: 

## CNN Model

In [ ]:
def build_cnn(input_shape):
    inp = keras.Input(shape=input_shape)
    x = layers.Conv1D(64, kernel_size=3, padding="causal", activation="relu")(inp)
    x = layers.Conv1D(64, kernel_size=3, padding="causal", activation="relu")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Conv1D(128, kernel_size=3, padding="causal", activation="relu")(x)
    x = layers.Conv1D(128, kernel_size=3, padding="causal", activation="relu")(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1)(x)

    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return model

model = build_cnn(X_train.shape[1:])
model.summary()

## Train

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=256,
    verbose=1
)

## Evaluate on test set (subset)

In [ ]:
X_test, y_test = make_dataset(
    test_ephem,
    test_era5_norm,
    test_wl,
    lat_vec,
    lon_vec,
    lookback=LOOKBACK,
    max_nodes=200
)

pred = model.predict(X_test, batch_size=256)
rmse = RMSE(y_test, pred.squeeze())
print("Test RMSE:", rmse)